# FIN42350 Capital Market - Group Project

This notebook implements the complete project in five phases. It deliberately keeps only the inputs, calculations, and charts needed to explain the portfolio decision.

1. Investor profile and risk assessment
2. Investor suitability and asset-class allocation
3. Market data preparation
4. Portfolio optimization
5. Benchmarking and performance analysis

All market returns and portfolio values are evaluated in euro. The final two years are reserved for out-of-sample testing.

In [3]:
from datetime import timedelta
from enum import StrEnum
from pathlib import Path
from typing import Annotated
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import yfinance as yf
from IPython.display import HTML, display
from pydantic import BaseModel, ConfigDict, Field, model_validator
from scipy.optimize import Bounds, minimize

# yfinance currently uses pandas inplace operations that Pandas 4 deprecates.
# Suppress only this third-party compatibility warning; retain all other warnings.
warnings.filterwarnings("ignore", category=getattr(pd.errors, "Pandas4Warning", DeprecationWarning), module=r"yfinance(?:\..*)?$")

pd.options.display.float_format = "{:,.4f}".format

TRADING_DAYS = 252
INITIAL_CAPITAL_EUR = 1_000_000.0
RISK_FREE_RATE_EUR = 0.0
RANDOM_SEED = 42_350


def show_figure(figure):
    """Render Plotly reliably in Jupyter and VS Code."""
    display(HTML(figure.to_html(include_plotlyjs="cdn", full_html=False)))


universe_candidates = [
    Path("asset_universe.csv"),
]
UNIVERSE_PATH = next((path for path in universe_candidates if path.exists()), None)
if UNIVERSE_PATH is None:
    raise FileNotFoundError("Could not locate asset_universe.csv")

asset_universe_df = pd.read_csv(UNIVERSE_PATH)
asset_universe_df["instrument_type"] = asset_universe_df["instrument_type"].str.upper()
asset_universe_df["asset_class"] = asset_universe_df["asset_class"].str.title()
asset_universe_df = asset_universe_df.rename(columns={"asset_subclass": "asset_type"})

required_columns = {
    "ticker", "display_name", "instrument_type",
    "asset_class", "asset_type", "geography",
}
missing = required_columns - set(asset_universe_df.columns)
if missing:
    raise ValueError(f"Missing asset-universe columns: {sorted(missing)}")

print(f"Loaded {len(asset_universe_df)} assets from {UNIVERSE_PATH}")

Loaded 134 assets from asset_universe.csv


## Phase I - Investor Profile and Risk Assessment

The questionnaire records five investor preferences. Each answer is converted to a score between 0 and 1, where a larger value consistently represents greater capacity or willingness to accept risk. The investor assigns ten importance points across the questions. Their weighted score determines the risk profile and the risk-aversion coefficient used by the optimizer.

In [4]:
class RiskTolerance(StrEnum):
    VERY_LOW = "Very low"
    LOW = "Low"
    MODERATE = "Moderate"
    HIGH = "High"
    VERY_HIGH = "Very high"


class InvestmentHorizon(StrEnum):
    VERY_SHORT = "Very short"  # Less than 1 year
    SHORT = "Short"            # 1–3 years
    MEDIUM = "Medium"          # 3–5 years
    LONG = "Long"              # 5–10 years
    VERY_LONG = "Very long"    # More than 10 years


class ObjectiveLevel(StrEnum):
    VERY_LOW = "Very low"
    LOW = "Low"
    MEDIUM = "Medium"
    HIGH = "High"
    VERY_HIGH = "Very high"


class InvestorProfile(BaseModel):
    model_config = ConfigDict(frozen=True, extra="forbid")

    risk_tolerance: RiskTolerance
    investment_horizon: InvestmentHorizon
    growth_objective: ObjectiveLevel
    capital_preservation: ObjectiveLevel
    liquidity_requirement: ObjectiveLevel


ImportancePoints = Annotated[int, Field(ge=0, le=10, strict=True)]


class QuestionWeights(BaseModel):
    model_config = ConfigDict(frozen=True, extra="forbid")

    risk_tolerance: ImportancePoints
    investment_horizon: ImportancePoints
    growth_objective: ImportancePoints
    capital_preservation: ImportancePoints
    liquidity_requirement: ImportancePoints

    @model_validator(mode="after")
    def total_must_equal_ten(self):
        if sum(self.model_dump().values()) != 10:
            raise ValueError("Importance points must sum to 10.")
        return self


# Illustrative client used for the project run.
investor = InvestorProfile(
    risk_tolerance=RiskTolerance.MODERATE,
    investment_horizon=InvestmentHorizon.LONG,
    growth_objective=ObjectiveLevel.HIGH,
    capital_preservation=ObjectiveLevel.MEDIUM,
    liquidity_requirement=ObjectiveLevel.LOW,
)
importance = QuestionWeights(
    risk_tolerance=3,
    investment_horizon=2,
    growth_objective=2,
    capital_preservation=2,
    liquidity_requirement=1,
)

In [5]:
increasing_scores = {
    "Very low": 0.00, "Low": 0.25, "Moderate": 0.50,
    "Medium": 0.50, "High": 0.75, "Very high": 1.00,
}
horizon_scores = {
    "Very short": 0.00, "Short": 0.25, "Medium": 0.50,
    "Long": 0.75, "Very long": 1.00,
}
decreasing_scores = {
    "Very low": 1.00, "Low": 0.75, "Medium": 0.50,
    "High": 0.25, "Very high": 0.00,
}
answer_scores = {
    "risk_tolerance": increasing_scores,
    "investment_horizon": horizon_scores,
    "growth_objective": increasing_scores,
    "capital_preservation": decreasing_scores,
    "liquidity_requirement": decreasing_scores,
}

answers = investor.model_dump(mode="json")
importance_points = importance.model_dump()
importance_weights = {key: value / 10 for key, value in importance_points.items()}

questionnaire_df = pd.DataFrame([
    {
        "question": key.replace("_", " ").title(),
        "answer": answer,
        "importance_points": importance_points[key],
        "normalized_weight": importance_weights[key],
        "normalized_answer": answer_scores[key][answer],
    }
    for key, answer in answers.items()
])
questionnaire_df["weighted_score"] = (
    questionnaire_df["normalized_weight"] * questionnaire_df["normalized_answer"]
)
investor_risk_score = float(questionnaire_df["weighted_score"].sum())

risk_profiles = [
    (0.20, "Very conservative", 12.0),
    (0.40, "Conservative", 8.0),
    (0.60, "Moderate", 5.0),
    (0.80, "Growth", 3.0),
    (1.01, "Aggressive", 1.0),
]
_, investor_risk_profile, risk_aversion_lambda = next(
    row for row in risk_profiles if investor_risk_score < row[0]
)

display(questionnaire_df)
print(f"Risk score: {investor_risk_score:.3f}")
print(f"Profile: {investor_risk_profile}")
print(f"Risk-aversion coefficient: {risk_aversion_lambda:.1f}")

,question,answer,importance_points,normalized_weight,normalized_answer,weighted_score
0,Risk Tolerance,Moderate,3,0.3000,0.5000,0.1500
1,Investment Horizon,Long,2,0.2000,0.7500,0.1500
2,Growth Objective,High,2,0.2000,0.7500,0.1500
3,Capital Preservation,Medium,2,0.2000,0.5000,0.1000
4,Liquidity Requirement,Low,1,0.1000,0.7500,0.0750


Risk score: 0.625
Profile: Growth
Risk-aversion coefficient: 3.0


## Phase II - Investor Suitability and Asset-Class Allocation

The suitability lookup maps each selected answer to a score for Equity, Fixed Income, Cash, Real Estate, and Commodity. The question weights combine those values into one suitability score per class. Normalizing the five scores produces strategic asset-class weights that sum to 100%.

In [6]:
ASSET_CLASSES = ("Equity", "Fixed Income", "Cash", "Real Estate", "Commodity")


def vector(equity, fixed_income, cash, real_estate, commodity):
    return dict(zip(
        ASSET_CLASSES,
        (equity, fixed_income, cash, real_estate, commodity),
        strict=True,
    ))


SUITABILITY = {
    "risk_tolerance": {
        "Very low": vector(0.10, 0.80, 1.00, 0.30, 0.20),
        "Low": vector(0.30, 0.90, 0.90, 0.40, 0.30),
        "Moderate": vector(0.60, 0.80, 0.60, 0.60, 0.50),
        "High": vector(0.85, 0.50, 0.30, 0.75, 0.70),
        "Very high": vector(1.00, 0.30, 0.10, 0.85, 0.85),
    },
    "investment_horizon": {
        "Very short": vector(0.10, 0.85, 1.00, 0.10, 0.10),
        "Short": vector(0.25, 0.90, 0.90, 0.25, 0.20),
        "Medium": vector(0.60, 0.80, 0.60, 0.55, 0.40),
        "Long": vector(1.00, 0.60, 0.20, 0.80, 0.60),
        "Very long": vector(1.00, 0.40, 0.10, 0.90, 0.75),
    },
    "growth_objective": {
        "Very low": vector(0.10, 0.90, 1.00, 0.20, 0.20),
        "Low": vector(0.30, 0.85, 0.85, 0.35, 0.30),
        "Medium": vector(0.60, 0.70, 0.55, 0.55, 0.40),
        "High": vector(0.95, 0.45, 0.15, 0.70, 0.55),
        "Very high": vector(1.00, 0.25, 0.05, 0.85, 0.70),
    },
    "capital_preservation": {
        "Very low": vector(0.95, 0.30, 0.10, 0.80, 0.75),
        "Low": vector(0.75, 0.55, 0.35, 0.70, 0.65),
        "Medium": vector(0.40, 0.85, 1.00, 0.50, 0.55),
        "High": vector(0.20, 0.95, 1.00, 0.30, 0.35),
        "Very high": vector(0.05, 1.00, 1.00, 0.10, 0.15),
    },
    "liquidity_requirement": {
        "Very low": vector(0.85, 0.80, 0.85, 0.70, 0.70),
        "Low": vector(0.80, 0.85, 0.95, 0.50, 0.60),
        "Medium": vector(0.65, 0.90, 1.00, 0.35, 0.45),
        "High": vector(0.45, 0.85, 1.00, 0.20, 0.30),
        "Very high": vector(0.20, 0.75, 1.00, 0.05, 0.10),
    },
}

In [7]:
selected_suitability_df = pd.DataFrame([
    {
        "question": key,
        "importance_weight": importance_weights[key],
        **SUITABILITY[key][answer],
    }
    for key, answer in answers.items()
])

weighted_scores = (
    selected_suitability_df[list(ASSET_CLASSES)]
    .mul(selected_suitability_df["importance_weight"], axis=0)
    .sum()
)
asset_class_allocation_df = pd.DataFrame({
    "asset_class": weighted_scores.index,
    "suitability_score": weighted_scores.values,
})
asset_class_allocation_df["target_weight"] = (
    asset_class_allocation_df["suitability_score"]
    / asset_class_allocation_df["suitability_score"].sum()
)
asset_class_targets = asset_class_allocation_df.set_index("asset_class")["target_weight"]

display(asset_class_allocation_df)

allocation_figure = px.pie(
    asset_class_allocation_df,
    names="asset_class",
    values="target_weight",
    title="Investor-specific asset-class allocation",
)
allocation_figure.update_traces(textinfo="label+percent")
show_figure(allocation_figure)

,asset_class,suitability_score,target_weight
0,Equity,0.7300,0.2310
1,Fixed Income,0.7050,0.2231
2,Cash,0.5450,0.1725
3,Real Estate,0.6300,0.1994
4,Commodity,0.5500,0.1741


## Phase III - Market Data Preparation

Seven years of adjusted daily prices are downloaded from Yahoo Finance. US-dollar prices are converted to euro using the daily USD/EUR rate before returns are calculated. The first five years form the in-sample estimation period; the most recent two years are held out for evaluation. Only assets with adequate history are retained.

In [8]:
END_DATE = pd.Timestamp.today().normalize()
START_DATE = END_DATE - pd.DateOffset(years=7)
OUT_OF_SAMPLE_START = END_DATE - pd.DateOffset(years=2)

tickers = asset_universe_df["ticker"].tolist()
download = yf.download(
    tickers,
    start=START_DATE,
    end=END_DATE + timedelta(days=1),
    auto_adjust=False,
    actions=False,
    progress=False,
    threads=True,
)
if download.empty:
    raise RuntimeError("Yahoo Finance returned no asset data.")

prices_usd = download["Adj Close"].copy()
prices_usd = prices_usd.reindex(columns=tickers)
prices_usd.columns.name = "ticker"
prices_usd.index = pd.to_datetime(prices_usd.index).tz_localize(None)

fx_download = yf.download(
    "USDEUR=X",
    start=START_DATE - timedelta(days=10),
    end=END_DATE + timedelta(days=1),
    auto_adjust=False,
    actions=False,
    progress=False,
)
if fx_download.empty:
    raise RuntimeError("Yahoo Finance returned no USD/EUR data.")

if isinstance(fx_download.columns, pd.MultiIndex):
    usd_eur = fx_download["Adj Close"].iloc[:, 0]
else:
    usd_eur = fx_download["Adj Close"]
usd_eur.index = pd.to_datetime(usd_eur.index).tz_localize(None)
usd_eur = usd_eur.reindex(prices_usd.index).ffill().bfill()

# Retain assets observed on at least 95% of in-sample trading dates.
in_sample_prices_usd = prices_usd.loc[prices_usd.index < OUT_OF_SAMPLE_START]
coverage = in_sample_prices_usd.notna().mean()
eligible_tickers = coverage[coverage >= 0.95].index.tolist()
if "SPY" not in eligible_tickers:
    raise ValueError("SPY lacks sufficient history for the benchmark.")

prices_eur = prices_usd[eligible_tickers].mul(usd_eur, axis=0).ffill(limit=3)
returns_eur = prices_eur.pct_change(fill_method=None)

in_sample_returns_df = (
    returns_eur.loc[returns_eur.index < OUT_OF_SAMPLE_START]
    .dropna(how="any")
)
annualized_returns = in_sample_returns_df.mean() * TRADING_DAYS
annualized_covariance_df = in_sample_returns_df.cov() * TRADING_DAYS
annualized_volatility = np.sqrt(np.diag(annualized_covariance_df))

asset_metrics_df = (
    asset_universe_df.set_index("ticker")
    .reindex(in_sample_returns_df.columns)
    .rename_axis("ticker")
    .assign(
        annualized_return=annualized_returns,
        annualized_volatility=annualized_volatility,
    )
    .reset_index()
)

print(f"Full period: {START_DATE.date()} to {END_DATE.date()}")
print(f"In-sample observations: {len(in_sample_returns_df):,}")
print(f"Eligible assets: {len(asset_metrics_df):,}")
display(asset_metrics_df.head())

Full period: 2019-10-24 to 2026-10-24
In-sample observations: 1,257
Eligible assets: 134


,ticker,display_name,instrument_type,asset_class,asset_type,geography,annualized_return,annualized_volatility
0,SPY,SPDR S&P 500 ETF Trust,ETF,Equity,US large blend,United States,0.1772,0.2202
1,IVV,iShares Core S&P 500 ETF,ETF,Equity,US large blend,United States,0.1784,0.2233
2,VOO,Vanguard S&P 500 ETF,ETF,Equity,US large blend,United States,0.1784,0.2229
3,VTI,Vanguard Total Stock Market ETF,ETF,Equity,US total market,United States,0.1724,0.2258
4,RSP,Invesco S&P 500 Equal Weight ETF,ETF,Equity,US large equal weight,United States,0.1500,0.2313


In [9]:
risk_return_figure = px.scatter(
    asset_metrics_df,
    x="annualized_volatility",
    y="annualized_return",
    color="asset_class",
    hover_name="ticker",
    title="In-sample annualized risk and return in EUR",
    labels={
        "annualized_volatility": "Annualized volatility",
        "annualized_return": "Annualized arithmetic return",
        "asset_class": "Asset class",
    },
)
show_figure(risk_return_figure)

## Phase IV - Portfolio Optimization

A long-only mean–variance optimizer selects ETFs within the five required asset classes. Each class must equal the Phase II target. A 15% ETF cap limits concentration, except where a class contains too few eligible ETFs to make that cap feasible. The investor risk-aversion coefficient controls the trade-off between expected return and variance.

In [10]:
BASE_ETF_CAP = 0.15

optimization_assets_df = (
    asset_metrics_df[
        asset_metrics_df["asset_class"].isin(ASSET_CLASSES)
    ]
    .sort_values(["asset_class", "ticker"])
    .reset_index(drop=True)
)
optimization_tickers = optimization_assets_df["ticker"].tolist()
mu = annualized_returns.reindex(optimization_tickers).to_numpy()
sigma = annualized_covariance_df.loc[
    optimization_tickers, optimization_tickers
].to_numpy()
sigma = (sigma + sigma.T) / 2
sigma += np.eye(len(sigma)) * 1e-10

class_labels = optimization_assets_df["asset_class"].to_numpy()
initial_weights = np.zeros(len(optimization_tickers))
upper_bounds = np.zeros(len(optimization_tickers))
constraints = []

for asset_class, target in asset_class_targets.items():
    mask = class_labels == asset_class
    count = int(mask.sum())
    if count == 0:
        raise ValueError(f"No eligible asset found for {asset_class}.")
    cap = max(BASE_ETF_CAP, float(target) / count)
    initial_weights[mask] = float(target) / count
    upper_bounds[mask] = cap
    constraints.append({
        "type": "eq",
        "fun": lambda w, m=mask, t=float(target): w[m].sum() - t,
    })


def negative_utility(weights):
    expected_return = weights @ mu
    variance = weights @ sigma @ weights
    return -(expected_return - 0.5 * risk_aversion_lambda * variance)


result = minimize(
    negative_utility,
    initial_weights,
    method="SLSQP",
    bounds=Bounds(np.zeros(len(mu)), upper_bounds),
    constraints=constraints,
    options={"ftol": 1e-12, "maxiter": 2_000},
)
if not result.success:
    raise RuntimeError(result.message)

optimal_weights = result.x
assert np.isclose(optimal_weights.sum(), 1.0, atol=1e-6)

optimized_portfolio_df = optimization_assets_df[
    ["ticker", "display_name", "instrument_type", "asset_class", "asset_type", "geography"]
].copy()
optimized_portfolio_df["optimal_weight"] = optimal_weights
optimized_portfolio_df["allocation_euro"] = optimal_weights * INITIAL_CAPITAL_EUR
selected_portfolio_df = (
    optimized_portfolio_df[optimized_portfolio_df["optimal_weight"] > 1e-5]
    .sort_values("optimal_weight", ascending=False)
    .reset_index(drop=True)
)

portfolio_expected_return = float(optimal_weights @ mu)
portfolio_variance = float(optimal_weights @ sigma @ optimal_weights)
portfolio_volatility = np.sqrt(portfolio_variance)
portfolio_sharpe = (
    (portfolio_expected_return - RISK_FREE_RATE_EUR) / portfolio_volatility
)

print(f"Expected annual return: {portfolio_expected_return:.2%}")
print(f"Annual volatility: {portfolio_volatility:.2%}")
print(f"Sharpe ratio: {portfolio_sharpe:.2f}")
display(selected_portfolio_df)

Expected annual return: 13.91%
Annual volatility: 15.31%
Sharpe ratio: 0.91


,ticker,display_name,instrument_type,asset_class,asset_type,geography,optimal_weight,allocation_euro
0,BIL,SPDR Bloomberg 1-3 Month T-Bill ETF,ETF,Cash,Treasury bills,United States,0.1725,"172,468.3544"
1,SMH,VanEck Semiconductor ETF,ETF,Equity,Semiconductors,Global,0.1500,"150,000.0000"
2,SCHP,Schwab US TIPS ETF,ETF,Fixed Income,Inflation-linked bonds,United States,0.1500,"150,000.0000"
3,XLRE,Real Estate Select Sector SPDR Fund,ETF,Real Estate,Listed real estate,United States,0.1500,"150,000.0000"
4,ARGT,Global X MSCI Argentina ETF,ETF,Equity,Country equity,Argentina,0.0810,"81,012.6582"
5,SLV,iShares Silver Trust,ETF,Commodity,Silver,Global,0.0764,"76,415.6769"
6,TIP,iShares TIPS Bond ETF,ETF,Fixed Income,Inflation-linked bonds,United States,0.0699,"69,878.4080"
7,IAU,iShares Gold Trust,ETF,Commodity,Gold,Global,0.0518,"51,781.5680"
8,VNQ,Vanguard Real Estate ETF,ETF,Real Estate,Listed real estate,United States,0.0494,"49,367.0886"
9,BNO,United States Brent Oil Fund,ETF,Commodity,Brent crude oil,Global,0.0459,"45,853.3881"


In [11]:
portfolio_figure = px.pie(
    selected_portfolio_df,
    names="ticker",
    values="optimal_weight",
    title="Optimized ETF allocation",
)
portfolio_figure.update_traces(textinfo="label+percent")
show_figure(portfolio_figure)

## Phase V - Portfolio Benchmarking and Performance Analysis

The optimized weights are frozen before the holdout period. One million euro is invested at the first common out-of-sample price and the ETF units are held without re-optimization. SPY is valued over the same dates as an investable S&P 500 benchmark. The comparison uses a compact set of return, risk, downside-risk, and terminal-wealth indicators.

In [12]:
selected_tickers = selected_portfolio_df["ticker"].tolist()
selected_weights = selected_portfolio_df.set_index("ticker")["optimal_weight"]
selected_weights = selected_weights / selected_weights.sum()

oos_prices_df = (
    prices_eur.loc[prices_eur.index >= OUT_OF_SAMPLE_START, selected_tickers]
    .dropna(how="any")
)
if oos_prices_df.empty:
    raise ValueError("No common out-of-sample prices are available.")

units = INITIAL_CAPITAL_EUR * selected_weights / oos_prices_df.iloc[0]
portfolio_value = oos_prices_df.mul(units, axis=1).sum(axis=1)

benchmark_price = prices_eur["SPY"].reindex(oos_prices_df.index).ffill()
benchmark_value = INITIAL_CAPITAL_EUR * benchmark_price / benchmark_price.iloc[0]

growth_df = pd.DataFrame({
    "Optimized portfolio": portfolio_value,
    "S&P 500 (SPY)": benchmark_value,
}).dropna()
growth_returns_df = growth_df.pct_change(fill_method=None).dropna()

def performance_metrics(values):
    returns = values.pct_change(fill_method=None).dropna()
    years = (values.index[-1] - values.index[0]).days / 365.25
    total_return = values.iloc[-1] / values.iloc[0] - 1
    annualized_return = (1 + total_return) ** (1 / years) - 1
    volatility = returns.std(ddof=1) * np.sqrt(TRADING_DAYS)
    drawdown = values / values.cummax() - 1
    return {
        "Total return": total_return,
        "Annualized return": annualized_return,
        "Annualized volatility": volatility,
        "Sharpe ratio": (annualized_return - RISK_FREE_RATE_EUR) / volatility,
        "Maximum drawdown": abs(drawdown.min()),
        "Ending value (EUR)": values.iloc[-1],
    }

performance_df = pd.DataFrame({
    name: performance_metrics(growth_df[name])
    for name in growth_df.columns
})

# Two benchmark-relative measures complete the comparison.
beta = (
    growth_returns_df["Optimized portfolio"].cov(growth_returns_df["S&P 500 (SPY)"])
    / growth_returns_df["S&P 500 (SPY)"].var(ddof=1)
)
alpha = (
    growth_returns_df["Optimized portfolio"].mean()
    - beta * growth_returns_df["S&P 500 (SPY)"].mean()
) * TRADING_DAYS
performance_df.loc["Beta"] = [beta, 1.0]
performance_df.loc["Jensen's alpha"] = [alpha, 0.0]

display(performance_df)

,Optimized portfolio,S&P 500 (SPY)
Total return,0.2997,0.2915
Annualized return,0.1404,0.1367
Annualized volatility,0.1583,0.1788
Sharpe ratio,0.8869,0.7647
Maximum drawdown,0.1399,0.2303
Ending value (EUR),"1,299,711.3617","1,291,498.1217"
Beta,0.6390,1.0000
Jensen's alpha,0.0520,0.0000


In [13]:
growth_plot_df = (
    growth_df.rename_axis("date")
    .reset_index()
    .melt(id_vars="date", var_name="strategy", value_name="value_euro")
)
growth_figure = px.line(
    growth_plot_df,
    x="date",
    y="value_euro",
    color="strategy",
    title="Out-of-sample growth of €1 million",
    labels={"value_euro": "Portfolio value (€)", "strategy": "Strategy"},
)
growth_figure.add_hline(
    y=INITIAL_CAPITAL_EUR,
    line_dash="dash",
    line_color="grey",
)
show_figure(growth_figure)

### Additional out-of-sample benchmarking figures

The following figures mirror the original notebook: a portfolio-only wealth path and a compact performance comparison table.

In [14]:
portfolio_growth_figure = px.line(
    x=growth_df.index,
    y=growth_df["Optimized portfolio"],
    title="Out-of-sample growth of €1 million: optimized portfolio",
    labels={"x": "Date", "y": "Portfolio value (€)"},
)
portfolio_growth_figure.add_hline(y=INITIAL_CAPITAL_EUR, line_dash="dash", line_color="grey")
show_figure(portfolio_growth_figure)

performance_table_df = performance_df.copy()
performance_table_df["Difference"] = (
    performance_table_df["Optimized portfolio"] - performance_table_df["S&P 500 (SPY)"]
)
percent_metrics = {"Total return", "Arithmetic annualized return", "Geometric annualized return", "Annualized volatility", "Maximum drawdown", "Jensen's alpha", "Residual risk", "M²", "R²"}
currency_metrics = {"Ending value"}
def format_performance_value(metric, value):
    if pd.isna(value):
        return "-"
    if metric in percent_metrics:
        return f"{value:.2%}"
    if metric in currency_metrics:
        return f"€{value:,.0f}"
    return f"{value:.3f}"

performance_table_figure = go.Figure(data=[go.Table(
    header=dict(values=["Metric", "Optimized portfolio", "S&P 500 (SPY)", "Difference"], fill_color="#D9EAF7", align="left"),
    cells=dict(values=[
        performance_table_df.index.tolist(),
        [format_performance_value(metric, value) for metric, value in performance_table_df["Optimized portfolio"].items()],
        [format_performance_value(metric, value) for metric, value in performance_table_df["S&P 500 (SPY)"].items()],
        [format_performance_value(metric, value) for metric, value in performance_table_df["Difference"].items()],
    ], fill_color="white", align="left"),
)])
performance_table_figure.update_layout(title="Out-of-sample performance comparison", height=520)
show_figure(performance_table_figure)

### Downside risk: historical VaR and Expected Shortfall

Historical one-day VaR is the loss threshold at the selected confidence level. Expected Shortfall is the average loss on observations at or beyond that threshold. Both are reported as positive losses.

In [15]:
def historical_var_es(returns, confidence):
    losses = -returns.dropna()
    value_at_risk = float(np.quantile(losses, confidence, method="higher"))
    expected_shortfall = float(losses[losses >= value_at_risk].mean())
    return value_at_risk, expected_shortfall

tail_risk_rows = []
for strategy in growth_returns_df:
    for confidence in (0.95, 0.99):
        var, es = historical_var_es(growth_returns_df[strategy], confidence)
        tail_risk_rows.append({
            "strategy": strategy,
            "confidence": confidence,
            "VaR": var,
            "Expected Shortfall": es,
            "VaR on €1m": var * INITIAL_CAPITAL_EUR,
            "ES on €1m": es * INITIAL_CAPITAL_EUR,
        })

tail_risk_df = pd.DataFrame(tail_risk_rows)
display(tail_risk_df)

portfolio_losses = -growth_returns_df["Optimized portfolio"]
loss_figure = px.histogram(
    x=portfolio_losses,
    nbins=40,
    title="Out-of-sample daily loss distribution",
    labels={"x": "Daily loss", "y": "Frequency"},
)
for confidence, color in [(0.95, "orange"), (0.99, "red")]:
    var = tail_risk_df.query(
        "strategy == 'Optimized portfolio' and confidence == @confidence"
    )["VaR"].iloc[0]
    es = tail_risk_df.query(
        "strategy == 'Optimized portfolio' and confidence == @confidence"
    )["Expected Shortfall"].iloc[0]
    loss_figure.add_vline(
        x=var,
        line_dash="dash",
        line_color=color,
        annotation_text=f"{confidence:.0%} VaR",
    )
    loss_figure.add_vline(
        x=es,
        line_dash="dot",
        line_color=color,
        annotation_text=f"{confidence:.0%} ES",
    )
show_figure(loss_figure)

,strategy,confidence,VaR,Expected Shortfall,VaR on €1m,ES on €1m
0,Optimized portfolio,0.9500,0.0146,0.0244,"14,623.2699","24,416.2049"
1,Optimized portfolio,0.9900,0.0293,0.0416,"29,320.1533","41,588.8589"
2,S&P 500 (SPY),0.9500,0.0164,0.0259,"16,355.6104","25,935.8578"
3,S&P 500 (SPY),0.9900,0.0277,0.0441,"27,742.2944","44,070.3189"


### Monte Carlo scenario analysis

A compact one-year simulation uses the in-sample portfolio and SPY means, volatilities, and correlation. It is a scenario analysis rather than a forecast. The output focuses on terminal wealth, probability of loss, and the probability that the portfolio exceeds SPY.

In [16]:
# Calibrate the two-strategy model from in-sample EUR returns.
calibration_df = pd.DataFrame({
    "Optimized portfolio": in_sample_returns_df[selected_tickers].dot(selected_weights),
    "S&P 500 (SPY)": in_sample_returns_df["SPY"],
}).dropna()

annual_mu = calibration_df.mean().to_numpy() * TRADING_DAYS
annual_cov = calibration_df.cov().to_numpy() * TRADING_DAYS
annual_cov = (annual_cov + annual_cov.T) / 2
annual_cov += np.eye(2) * 1e-12
cholesky = np.linalg.cholesky(annual_cov)

SCENARIOS = 5_000
DAYS = 252
rng = np.random.default_rng(RANDOM_SEED)
shocks = rng.standard_normal((SCENARIOS, DAYS, 2))
daily_log_returns = (
    (annual_mu - 0.5 * np.diag(annual_cov)) / TRADING_DAYS
    + (shocks @ cholesky.T) / np.sqrt(TRADING_DAYS)
)
simulated_values = INITIAL_CAPITAL_EUR * np.exp(
    np.cumsum(daily_log_returns, axis=1)
)
simulated_values = np.concatenate(
    [np.full((SCENARIOS, 1, 2), INITIAL_CAPITAL_EUR), simulated_values],
    axis=1,
)
terminal_values = simulated_values[:, -1, :]
terminal_returns = terminal_values / INITIAL_CAPITAL_EUR - 1

simulation_summary_df = pd.DataFrame({
    "strategy": calibration_df.columns,
    "median_terminal_wealth_euro": np.median(terminal_values, axis=0),
    "terminal_wealth_p05_euro": np.quantile(terminal_values, 0.05, axis=0),
    "terminal_wealth_p95_euro": np.quantile(terminal_values, 0.95, axis=0),
    "probability_of_loss": np.mean(terminal_returns < 0, axis=0),
})
probability_beats_spy = np.mean(terminal_values[:, 0] > terminal_values[:, 1])

display(simulation_summary_df)
print(f"Probability optimized portfolio beats SPY: {probability_beats_spy:.2%}")

,strategy,median_terminal_wealth_euro,terminal_wealth_p05_euro,terminal_wealth_p95_euro,probability_of_loss
0,Optimized portfolio,"1,137,087.5567","884,695.0278","1,459,735.1418",0.1960
1,S&P 500 (SPY),"1,165,363.2918","818,803.4399","1,670,337.4074",0.2388


Probability optimized portfolio beats SPY: 40.98%


In [17]:
days = np.arange(DAYS + 1)
simulation_figure = go.Figure()
colors = ("royalblue", "darkorange")

for index, name in enumerate(calibration_df.columns):
    median = np.median(simulated_values[:, :, index], axis=0)
    lower = np.quantile(simulated_values[:, :, index], 0.05, axis=0)
    upper = np.quantile(simulated_values[:, :, index], 0.95, axis=0)
    color = colors[index]

    simulation_figure.add_trace(go.Scatter(
        x=days, y=upper, mode="lines", line={"width": 0},
        showlegend=False, hoverinfo="skip",
    ))
    simulation_figure.add_trace(go.Scatter(
        x=days, y=lower, mode="lines", line={"width": 0},
        fill="tonexty", opacity=0.15, name=f"{name} 5th–95th percentile",
        hoverinfo="skip",
    ))
    simulation_figure.add_trace(go.Scatter(
        x=days, y=median, mode="lines", name=f"{name} median",
        line={"color": color},
    ))

simulation_figure.update_layout(
    title="One-year Monte Carlo wealth scenarios",
    xaxis_title="Trading day",
    yaxis_title="Portfolio value (€)",
)
show_figure(simulation_figure)

### Terminal-return distribution and simulated performance

These figures complete the simulation view by comparing terminal-return distributions and the median performance indicators across all simulated paths.

In [18]:
terminal_distribution_df = pd.DataFrame(terminal_returns, columns=calibration_df.columns).melt(var_name="strategy", value_name="terminal_return")
terminal_distribution_figure = px.histogram(
    terminal_distribution_df, x="terminal_return", color="strategy", nbins=60,
    barmode="overlay", opacity=0.55, title="One-year simulated terminal-return distribution",
    labels={"terminal_return": "Terminal return", "strategy": "Strategy"},
)
terminal_distribution_figure.update_xaxes(tickformat=".0%")
show_figure(terminal_distribution_figure)

simulated_simple_returns = np.exp(daily_log_returns) - 1
annualized_returns = simulated_simple_returns.mean(axis=1) * TRADING_DAYS
annualized_volatility = simulated_simple_returns.std(axis=1, ddof=1) * np.sqrt(TRADING_DAYS)
sharpe_ratio = np.divide(annualized_returns, annualized_volatility, out=np.full_like(annualized_returns, np.nan), where=annualized_volatility > 0)
running_max = np.maximum.accumulate(simulated_values, axis=1)
maximum_drawdown = np.max(1 - simulated_values / running_max, axis=1)
portfolio_daily = simulated_simple_returns[:, :, 0]
benchmark_daily = simulated_simple_returns[:, :, 1]
portfolio_centered = portfolio_daily - portfolio_daily.mean(axis=1, keepdims=True)
benchmark_centered = benchmark_daily - benchmark_daily.mean(axis=1, keepdims=True)
covariance_paths = np.sum(portfolio_centered * benchmark_centered, axis=1) / (DAYS - 1)
benchmark_variance_paths = np.var(benchmark_daily, axis=1, ddof=1)
beta_paths = covariance_paths / benchmark_variance_paths
correlation_paths = covariance_paths / (portfolio_daily.std(axis=1, ddof=1) * benchmark_daily.std(axis=1, ddof=1))
alpha_paths = annualized_returns[:, 0] - beta_paths * annualized_returns[:, 1]
residual_daily = portfolio_centered - beta_paths[:, None] * benchmark_centered
residual_risk_paths = residual_daily.std(axis=1, ddof=1) * np.sqrt(TRADING_DAYS)
active_daily = portfolio_daily - benchmark_daily
tracking_error_paths = active_daily.std(axis=1, ddof=1) * np.sqrt(TRADING_DAYS)
information_ratio_paths = np.divide(active_daily.mean(axis=1) * TRADING_DAYS, tracking_error_paths, out=np.full(SCENARIOS, np.nan), where=tracking_error_paths > 0)
treynor_paths = np.divide(annualized_returns[:, 0], beta_paths, out=np.full(SCENARIOS, np.nan), where=np.abs(beta_paths) > 1e-12)
m2_paths = sharpe_ratio[:, 0] * annualized_volatility[:, 1] - annualized_returns[:, 1]

simulated_metric_paths = {
    "Total return": terminal_returns,
    "Arithmetic annualized return": annualized_returns,
    "Geometric annualized return": terminal_returns,
    "Annualized volatility": annualized_volatility,
    "Sharpe ratio": sharpe_ratio,
    "Maximum drawdown": maximum_drawdown,
    "Beta vs S&P 500": np.column_stack([beta_paths, np.ones(SCENARIOS)]),
    "Jensen's alpha": np.column_stack([alpha_paths, np.zeros(SCENARIOS)]),
    "Treynor ratio": np.column_stack([treynor_paths, annualized_returns[:, 1]]),
    "Residual risk": np.column_stack([residual_risk_paths, np.zeros(SCENARIOS)]),
    "Information ratio": np.column_stack([information_ratio_paths, np.full(SCENARIOS, np.nan)]),
    "M²": np.column_stack([m2_paths, np.zeros(SCENARIOS)]),
    "R²": np.column_stack([correlation_paths ** 2, np.ones(SCENARIOS)]),
}
def finite_median(values):
    finite_values = values[np.isfinite(values)]
    return float(np.median(finite_values)) if finite_values.size else np.nan

simulation_performance_df = pd.DataFrame({
    strategy: {metric: finite_median(values[:, index]) for metric, values in simulated_metric_paths.items()}
    for index, strategy in enumerate(calibration_df.columns)
})
simulation_performance_df["Difference"] = simulation_performance_df.iloc[:, 0] - simulation_performance_df.iloc[:, 1]
simulation_performance_table_figure = go.Figure(data=[go.Table(
    header=dict(values=["Metric", *simulation_performance_df.columns], fill_color="#D9EAF7", align="left"),
    cells=dict(values=[
        simulation_performance_df.index.tolist(),
        *[[format_performance_value(metric, value) for metric, value in simulation_performance_df[column].items()] for column in simulation_performance_df.columns],
    ], fill_color="white", align="left"),
)])
simulation_performance_table_figure.update_layout(title="Median simulated performance indicators", height=600)
show_figure(simulation_performance_table_figure)

### Save the essential outputs

Only the tables required for the report and reproducibility are persisted.

In [19]:
OUTPUT_DIR = UNIVERSE_PATH.parent / "output" / "five_phase_workflow"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

market_data_daily_eur_df = (
    prices_eur.rename_axis("date")
    .reset_index()
    .melt(id_vars="date", var_name="ticker", value_name="adjusted_close_euro")
)
daily_returns_long_df = (
    returns_eur.rename_axis("date")
    .reset_index()
    .melt(id_vars="date", var_name="ticker", value_name="daily_return_euro")
)
market_data_daily_eur_df = market_data_daily_eur_df.merge(
    daily_returns_long_df, on=["date", "ticker"], how="left"
)
market_data_daily_eur_df.to_pickle(OUTPUT_DIR / "market_data_daily_eur.pkl")
asset_class_allocation_df.to_csv(OUTPUT_DIR / "asset_class_allocation.csv", index=False)
asset_metrics_df.to_pickle(OUTPUT_DIR / "asset_metrics.pkl")
annualized_covariance_df.to_pickle(OUTPUT_DIR / "annualized_covariance.pkl")
selected_portfolio_df.to_csv(OUTPUT_DIR / "optimized_portfolio.csv", index=False)
growth_df.to_pickle(OUTPUT_DIR / "out_of_sample_growth.pkl")
performance_df.to_csv(OUTPUT_DIR / "performance_comparison.csv")
tail_risk_df.to_csv(OUTPUT_DIR / "historical_var_es.csv", index=False)
simulation_summary_df.to_csv(OUTPUT_DIR / "simulation_summary.csv", index=False)

print(f"Saved outputs to {OUTPUT_DIR}")

Saved outputs to output/five_phase_workflow


## Three Investor Scenarios - Risky, Balanced, and Conservative

The same five-phase workflow is applied to three contrasting investors. Only the phase outputs and brief interpretations are shown; all scenarios use the same market-data sample and €1 million initial capital.

In [20]:
scenario_definitions = {
    "Risky investor": (
        InvestorProfile(
            risk_tolerance=RiskTolerance.VERY_HIGH,
            investment_horizon=InvestmentHorizon.VERY_LONG,
            growth_objective=ObjectiveLevel.VERY_HIGH,
            capital_preservation=ObjectiveLevel.LOW,
            liquidity_requirement=ObjectiveLevel.LOW,
        ),
        QuestionWeights(risk_tolerance=3, investment_horizon=2, growth_objective=3, capital_preservation=1, liquidity_requirement=1),
    ),
    "Balanced investor": (
        InvestorProfile(
            risk_tolerance=RiskTolerance.MODERATE,
            investment_horizon=InvestmentHorizon.MEDIUM,
            growth_objective=ObjectiveLevel.MEDIUM,
            capital_preservation=ObjectiveLevel.MEDIUM,
            liquidity_requirement=ObjectiveLevel.MEDIUM,
        ),
        QuestionWeights(risk_tolerance=3, investment_horizon=2, growth_objective=2, capital_preservation=2, liquidity_requirement=1),
    ),
    "Conservative investor": (
        InvestorProfile(
            risk_tolerance=RiskTolerance.LOW,
            investment_horizon=InvestmentHorizon.SHORT,
            growth_objective=ObjectiveLevel.LOW,
            capital_preservation=ObjectiveLevel.HIGH,
            liquidity_requirement=ObjectiveLevel.HIGH,
        ),
        QuestionWeights(risk_tolerance=3, investment_horizon=2, growth_objective=1, capital_preservation=3, liquidity_requirement=1),
    ),
}

def scenario_risk_output(profile, weights):
    scenario_answers = profile.model_dump(mode="json")
    points = weights.model_dump()
    normalized_weights = {key: value / 10 for key, value in points.items()}
    questionnaire = pd.DataFrame([
        {
            "question": key,
            "answer": answer,
            "importance_points": points[key],
            "normalized_weight": normalized_weights[key],
            "normalized_answer": answer_scores[key][answer],
        }
        for key, answer in scenario_answers.items()
    ])
    questionnaire["weighted_score"] = questionnaire["normalized_weight"] * questionnaire["normalized_answer"]
    score = float(questionnaire["weighted_score"].sum())
    _, profile_name, risk_lambda = next(row for row in risk_profiles if score < row[0])
    return scenario_answers, normalized_weights, questionnaire, score, profile_name, risk_lambda

def scenario_allocation_output(scenario_answers, normalized_weights):
    suitability = pd.DataFrame([
        {"question": key, "importance_weight": normalized_weights[key], **SUITABILITY[key][answer]}
        for key, answer in scenario_answers.items()
    ])
    scores = suitability[list(ASSET_CLASSES)].mul(suitability["importance_weight"], axis=0).sum()
    allocation = pd.DataFrame({"asset_class": scores.index, "suitability_score": scores.values})
    allocation["target_weight"] = allocation["suitability_score"] / allocation["suitability_score"].sum()
    return suitability, allocation

def scenario_optimization_output(allocation, risk_lambda):
    targets = allocation.set_index("asset_class")["target_weight"]
    start_weights = np.zeros(len(optimization_tickers))
    caps = np.zeros(len(optimization_tickers))
    scenario_constraints = []
    for asset_class, target in targets.items():
        mask = class_labels == asset_class
        count = int(mask.sum())
        cap = max(BASE_ETF_CAP, float(target) / count)
        start_weights[mask] = float(target) / count
        caps[mask] = cap
        scenario_constraints.append({"type": "eq", "fun": lambda w, m=mask, t=float(target): w[m].sum() - t})

    def scenario_negative_utility(weights):
        return -(weights @ mu - 0.5 * risk_lambda * (weights @ sigma @ weights))

    scenario_result = minimize(
        scenario_negative_utility, start_weights, method="SLSQP",
        bounds=Bounds(np.zeros(len(mu)), caps), constraints=scenario_constraints,
        options={"ftol": 1e-12, "maxiter": 2_000},
    )
    if not scenario_result.success:
        raise RuntimeError(scenario_result.message)
    weights = scenario_result.x
    holdings = optimization_assets_df[["ticker", "display_name", "asset_class", "asset_type", "geography"]].copy()
    holdings["optimal_weight"] = weights
    holdings["allocation_euro"] = weights * INITIAL_CAPITAL_EUR
    holdings = holdings[holdings["optimal_weight"] > 1e-5].sort_values("optimal_weight", ascending=False).reset_index(drop=True)
    expected_return = float(weights @ mu)
    volatility = float(np.sqrt(weights @ sigma @ weights))
    summary = {
        "expected_annual_return": expected_return,
        "annualized_volatility": volatility,
        "sharpe_ratio": (expected_return - RISK_FREE_RATE_EUR) / volatility,
        "selected_assets": len(holdings),
    }
    return weights, holdings, summary

def scenario_oos_output(holdings):
    tickers = holdings["ticker"].tolist()
    weights = holdings.set_index("ticker")["optimal_weight"]
    weights = weights / weights.sum()
    prices = prices_eur.loc[prices_eur.index >= OUT_OF_SAMPLE_START, tickers].dropna(how="any")
    units = INITIAL_CAPITAL_EUR * weights / prices.iloc[0]
    return prices.mul(units, axis=1).sum(axis=1)

scenario_results = {}
for scenario_name, (scenario_profile, scenario_weights) in scenario_definitions.items():
    scenario_answers, normalized_weights, questionnaire, score, profile_name, risk_lambda = scenario_risk_output(scenario_profile, scenario_weights)
    suitability, allocation = scenario_allocation_output(scenario_answers, normalized_weights)
    weights, holdings, optimization_summary = scenario_optimization_output(allocation, risk_lambda)
    wealth = scenario_oos_output(holdings)
    scenario_results[scenario_name] = {
        "answers": scenario_answers, "questionnaire": questionnaire, "risk_score": score,
        "risk_profile": profile_name, "risk_aversion": risk_lambda, "suitability": suitability,
        "allocation": allocation, "weights": weights, "holdings": holdings,
        "optimization_summary": optimization_summary, "wealth": wealth,
    }

### Phase I - Investor Profile and Risk Assessment

The profiles produce the intended progression from high willingness to accept risk to strong capital-preservation preferences.

In [21]:
scenario_profile_df = pd.DataFrame([
    {"scenario": name, **result["answers"], "risk_score": result["risk_score"], "risk_profile": result["risk_profile"], "risk_aversion_lambda": result["risk_aversion"]}
    for name, result in scenario_results.items()
])
display(scenario_profile_df)

,scenario,risk_tolerance,investment_horizon,growth_objective,capital_preservation,liquidity_requirement,risk_score,risk_profile,risk_aversion_lambda
0,Risky investor,Very high,Very long,Very high,Low,Low,0.9500,Aggressive,1.0000
1,Balanced investor,Moderate,Medium,Medium,Medium,Medium,0.5000,Moderate,5.0000
2,Conservative investor,Low,Short,Low,High,High,0.2500,Conservative,8.0000


### Phase II - Investor Suitability and Asset-Class Allocation

The risky investor receives the largest growth-asset allocation, whereas the conservative investor receives greater fixed-income and cash exposure.

In [22]:
scenario_allocation_df = pd.concat([
    result["allocation"].assign(scenario=name)
    for name, result in scenario_results.items()
], ignore_index=True)
display(scenario_allocation_df.pivot(index="asset_class", columns="scenario", values="target_weight"))
scenario_allocation_figure = px.bar(
    scenario_allocation_df, x="asset_class", y="target_weight", color="scenario",
    barmode="group", title="Asset-class allocation by investor scenario",
    labels={"asset_class": "Asset class", "target_weight": "Target weight", "scenario": "Scenario"},
)
scenario_allocation_figure.update_yaxes(tickformat=".0%")
show_figure(scenario_allocation_figure)

scenario,Balanced investor,Conservative investor,Risky investor
asset_class,,,
Cash,0.2309,0.3431,0.0632
Commodity,0.1512,0.1083,0.2399
Equity,0.1837,0.1009,0.3096
Fixed Income,0.2602,0.3321,0.1248
Real Estate,0.1740,0.1156,0.2626


### Phase III - Market Data Preparation

All scenarios use the same EUR-denominated observations, ensuring that differences arise from investor preferences rather than different market samples.

In [23]:
scenario_market_data_df = pd.DataFrame([
    {"dataset": "Full sample", "start": START_DATE.date(), "end": END_DATE.date(), "observations": len(returns_eur)},
    {"dataset": "In-sample", "start": START_DATE.date(), "end": (OUT_OF_SAMPLE_START - pd.Timedelta(days=1)).date(), "observations": len(in_sample_returns_df)},
    {"dataset": "Out-of-sample", "start": OUT_OF_SAMPLE_START.date(), "end": END_DATE.date(), "observations": int((returns_eur.index >= OUT_OF_SAMPLE_START).sum())},
])
scenario_market_data_df["eligible_assets"] = len(optimization_tickers)
display(scenario_market_data_df)

,dataset,start,end,observations,eligible_assets
0,Full sample,2019-10-24,2026-10-24,1761,119
1,In-sample,2019-10-24,2024-10-23,1257,119
2,Out-of-sample,2024-10-24,2026-10-24,503,119


### Phase IV - Portfolio Optimization

The optimizer selects ETFs within each scenario's asset-class constraints and applies the corresponding risk-aversion coefficient.

In [24]:
scenario_optimization_df = pd.DataFrame([
    {"scenario": name, **result["optimization_summary"]}
    for name, result in scenario_results.items()
])
display(scenario_optimization_df)
scenario_holdings_df = pd.concat([
    result["holdings"].assign(scenario=name)
    for name, result in scenario_results.items()
], ignore_index=True)
display(scenario_holdings_df[["scenario", "display_name", "asset_class", "asset_type", "geography", "optimal_weight", "allocation_euro"]])

,scenario,expected_annual_return,annualized_volatility,sharpe_ratio,selected_assets
0,Risky investor,0.1782,0.2061,0.8648,9
1,Balanced investor,0.1160,0.1259,0.9215,11
2,Conservative investor,0.0814,0.0973,0.8366,11


,scenario,display_name,asset_class,asset_type,geography,optimal_weight,allocation_euro
0,Risky investor,VanEck Semiconductor ETF,Equity,Semiconductors,Global,0.1500,"150,000.0000"
1,Risky investor,Real Estate Select Sector SPDR Fund,Real Estate,Listed real estate,United States,0.1500,"150,000.0000"
2,Risky investor,iShares Silver Trust,Commodity,Silver,Global,0.1488,"148,821.8841"
3,Risky investor,Global X MSCI Argentina ETF,Equity,Country equity,Argentina,0.1288,"128,759.4577"
4,Risky investor,SPDR Bloomberg High Yield Bond ETF,Fixed Income,High yield corporate,United States,0.1248,"124,797.4068"
5,Risky investor,Vanguard Real Estate ETF,Real Estate,Listed real estate,United States,0.1126,"112,560.7780"
6,Risky investor,United States Brent Oil Fund,Commodity,Brent crude oil,Global,0.0910,"91,048.4562"
7,Risky investor,SPDR Bloomberg 1-3 Month T-Bill ETF,Cash,Treasury bills,United States,0.0632,"63,209.0762"
8,Risky investor,iShares Semiconductor ETF,Equity,Semiconductors,United States,0.0308,"30,802.9410"
9,Balanced investor,SPDR Bloomberg 1-3 Month T-Bill ETF,Cash,Treasury bills,United States,0.2309,"230,894.3089"


### Phase V - Portfolio Benchmarking and Performance Analysis

Out-of-sample results compare each frozen portfolio with SPY. The table reports performance and tail risk; the chart shows the growth of the initial €1 million.

In [25]:
scenario_growth_df = pd.concat(
    {name: result["wealth"] for name, result in scenario_results.items()}, axis=1,
).dropna()
scenario_benchmark = prices_eur["SPY"].reindex(scenario_growth_df.index).ffill()
scenario_growth_df["S&P 500 (SPY)"] = INITIAL_CAPITAL_EUR * scenario_benchmark / scenario_benchmark.iloc[0]
scenario_performance_df = pd.DataFrame({
    strategy: performance_metrics(scenario_growth_df[strategy])
    for strategy in scenario_growth_df.columns
})
display(scenario_performance_df)

scenario_growth_plot_df = scenario_growth_df.rename_axis("date").reset_index().melt(id_vars="date", var_name="strategy", value_name="value_euro")
scenario_growth_figure = px.line(
    scenario_growth_plot_df, x="date", y="value_euro", color="strategy",
    title="Out-of-sample growth of €1 million by investor scenario",
    labels={"value_euro": "Portfolio value (€)", "strategy": "Strategy"},
)
scenario_growth_figure.add_hline(y=INITIAL_CAPITAL_EUR, line_dash="dash", line_color="grey")
show_figure(scenario_growth_figure)

scenario_returns_df = scenario_growth_df.pct_change(fill_method=None).dropna()
scenario_tail_risk_df = pd.DataFrame([
    {"strategy": strategy, "confidence": confidence, "VaR": var, "Expected Shortfall": es, "VaR on €1m": var * INITIAL_CAPITAL_EUR, "ES on €1m": es * INITIAL_CAPITAL_EUR}
    for strategy in scenario_returns_df.columns
    for confidence in (0.95, 0.99)
    for var, es in [historical_var_es(scenario_returns_df[strategy], confidence)]
])
display(scenario_tail_risk_df)

,Risky investor,Balanced investor,Conservative investor,S&P 500 (SPY)
Total return,0.4155,0.2403,0.1645,0.2915
Annualized return,0.1902,0.1139,0.0793,0.1367
Annualized volatility,0.2087,0.1265,0.1010,0.1788
Sharpe ratio,0.9111,0.9008,0.7852,0.7647
Maximum drawdown,0.1895,0.1226,0.1157,0.2303
Ending value (EUR),"1,415,514.0875","1,240,281.9216","1,164,539.9439","1,291,498.1217"


,strategy,confidence,VaR,Expected Shortfall,VaR on €1m,ES on €1m
0,Risky investor,0.9500,0.0188,0.0329,"18,803.5345","32,864.5057"
1,Risky investor,0.9900,0.0415,0.0596,"41,482.6110","59,582.5186"
2,Balanced investor,0.9500,0.0120,0.0183,"12,030.3197","18,281.3059"
3,Balanced investor,0.9900,0.0253,0.0290,"25,341.3654","29,036.6515"
4,Conservative investor,0.9500,0.0099,0.0142,"9,933.0041","14,205.3784"
5,Conservative investor,0.9900,0.0207,0.0226,"20,699.9552","22,593.1183"
6,S&P 500 (SPY),0.9500,0.0164,0.0259,"16,355.6104","25,935.8578"
7,S&P 500 (SPY),0.9900,0.0277,0.0441,"27,742.2944","44,070.3189"
